# PPO Agent Training with Loss Plotting
This notebook trains a PPO agent and plots training/evaluation loss curves.

In [ ]:
# Importing libraries
import gymnasium as gym
import robosuite as robs
import robosuite.wrappers.gym_wrapper as GymWrapper
from gym_wrapper import RobosuiteGymWrapper
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.evaluation import evaluate_policy
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Custom callback to track training metrics
class TrainingPlotCallback(BaseCallback):
    def __init__(self, eval_env, eval_freq=1000, n_eval_episodes=5, verbose=0):
        super(TrainingPlotCallback, self).__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.n_eval_episodes = n_eval_episodes
        
        # Storage for metrics
        self.train_losses = []
        self.eval_losses = []
        self.train_iterations = []
        self.eval_iterations = []
        self.eval_rewards = []
        
    def _on_step(self) -> bool:
        # Log training loss
        if len(self.model.logger.name_to_value) > 0:
            if 'train/loss' in self.model.logger.name_to_value:
                self.train_losses.append(self.model.logger.name_to_value['train/loss'])
                self.train_iterations.append(self.num_timesteps)
        
        # Periodic evaluation
        if self.num_timesteps % self.eval_freq == 0:
            # Evaluate the model
            episode_rewards = []
            for _ in range(self.n_eval_episodes):
                obs, _ = self.eval_env.reset()
                done = False
                episode_reward = 0
                while not done:
                    action, _ = self.model.predict(obs, deterministic=True)
                    obs, reward, terminated, truncated, _ = self.eval_env.step(action)
                    done = terminated or truncated
                    episode_reward += reward
                episode_rewards.append(episode_reward)
            
            # Store evaluation metrics
            mean_reward = np.mean(episode_rewards)
            eval_loss = -mean_reward  # Using negative mean reward as proxy for loss
            self.eval_losses.append(eval_loss)
            self.eval_rewards.append(mean_reward)
            self.eval_iterations.append(self.num_timesteps)
            
            if self.verbose > 0:
                print(f"Eval at step {self.num_timesteps}: Mean Reward = {mean_reward:.2f}, Eval Loss = {eval_loss:.2f}")
        
        return True

In [ ]:
# Create environment
env = RobosuiteGymWrapper("Lift", robots="Panda", has_renderer=False, use_camera_obs=False, reward_shaping=True)

# Option to load a pre-trained model or start fresh
# model = PPO.load("ppo_lift_model", env=env)  # Uncomment to load trained model
model = PPO("MlpPolicy", env, verbose=1)

In [ ]:
# Create callback instance
callback = TrainingPlotCallback(eval_env=env, eval_freq=500, n_eval_episodes=5, verbose=1)

In [ ]:
# Train the model with callback
model.learn(total_timesteps=10000, callback=callback)

In [ ]:
# Plot Training and Evaluation Loss
plt.figure(figsize=(14, 6))

# Subplot 1: Loss curves
plt.subplot(1, 2, 1)
if len(callback.train_losses) > 0:
    plt.plot(callback.train_iterations, callback.train_losses, 
             label='Training Loss', color='blue', alpha=0.6, linewidth=2)

if len(callback.eval_losses) > 0:
    plt.plot(callback.eval_iterations, callback.eval_losses, 
             label='Evaluation Loss', color='red', marker='o', 
             linewidth=2, markersize=6)

plt.xlabel('Training Iteration (Timesteps)', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training and Evaluation Loss vs. Iteration', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

# Subplot 2: Evaluation rewards
plt.subplot(1, 2, 2)
if len(callback.eval_rewards) > 0:
    plt.plot(callback.eval_iterations, callback.eval_rewards, 
             label='Mean Evaluation Reward', color='green', marker='s', 
             linewidth=2, markersize=6)

plt.xlabel('Training Iteration (Timesteps)', fontsize=12)
plt.ylabel('Mean Reward', fontsize=12)
plt.title('Evaluation Reward vs. Iteration', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\n{'='*60}")
print(f"Training Summary:")
print(f"{'='*60}")
print(f"Total training iterations: {len(callback.train_losses)}")
print(f"Total evaluation points: {len(callback.eval_losses)}")
if len(callback.train_losses) > 0:
    print(f"Final training loss: {callback.train_losses[-1]:.4f}")
if len(callback.eval_losses) > 0:
    print(f"Final evaluation loss: {callback.eval_losses[-1]:.4f}")
    print(f"Best evaluation loss: {min(callback.eval_losses):.4f}")
if len(callback.eval_rewards) > 0:
    print(f"Final mean reward: {callback.eval_rewards[-1]:.4f}")
    print(f"Best mean reward: {max(callback.eval_rewards):.4f}")
print(f"{'='*60}")

In [ ]:
# Save the trained model
model.save("ppo_lift_model")
print("Model saved successfully!")

In [ ]:
# Final evaluation with rendering (optional)
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=10, render=False)
print(f"\nFinal Evaluation:")
print(f"Mean Reward: {mean_reward:.2f} +/- {std_reward:.2f}")